In [1]:
from typing import List 
import struct
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt 
import sqlite3 , sqlite_vec

c:\Users\ADMIN\OneDrive\Desktop\Trial_Gen_ai\hands on RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_dimension = model.get_sentence_embedding_dimension()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7669.47it/s]
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14964\971044744.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dimension = model.get_sentence_embedding_dimension()


In [3]:
# Prepare Sqlite database-----------first we on the db for extrenal extension and then off after adding that 
sqlite_db = sqlite3.connect("embeddings.db")
sqlite_db.enable_load_extension(True)
sqlite_vec.load(sqlite_db)
sqlite_db.enable_load_extension(False)

In [5]:
sentences = [
    "I am a happy person.",
    "I am a joyful person.",
    "I am a pessimistic person.",
    "I am not an optimistic person."
]

embeddings = model.encode(sentences , normalize_embeddings=True)

In [6]:
# Create Database and table 
sqlite_db.execute("DROP TABLE IF EXISTS chunks")
sqlite_create_table = f"CREATE VIRTUAL TABLE chunks USING vec0(chunk_id INTEGER PRIMARY KEY , text TEXT , embedding float[{embedding_dimension}])"
sqlite_db.execute(sqlite_create_table)
sqlite_db.commit()

In [7]:
# Insert embeddings

def serialize_f32(vector : List[float]) -> bytes:
    """Serialize a list of floats into a compact "way bytes" format"""
    return struct.pack("%sf" % len(vector) , *vector)

for i , (text, embedding) in enumerate(zip(sentences , embeddings)):
    sql_insert = f"INSERT INTO chunks (chunk_id,text,embedding) VALUES (?,?,?)"
    sqlite_db.execute(sql_insert , (i , text, serialize_f32(embedding)))

sqlite_db.commit()

In [8]:
# Search using the embedding of the new sentence
query = "I am a glad person."
query_embedding = model.encode(query , normalize_embeddings= True)

sql_select = f"SELECT chunk_id , text , embedding , distance FROM chunks WHERE embedding MATCH '{query_embedding.tolist()}' ORDER BY distance LIMIT 1"
cursor = sqlite_db.execute(sql_select)
results = cursor.fetchall()

for chunk_id , text , embedding , distance in results:
    print(f"Chunk ID : {chunk_id} , Text : {text} , Distance : {distance}")

Chunk ID : 0 , Text : I am a happy person. , Distance : 0.7883185744285583


In [9]:
# Search using the embedding of the new sentence
query = "I am a negative person."
query_embedding = model.encode(query , normalize_embeddings= True)

sql_select = f"SELECT chunk_id , text , embedding , distance FROM chunks WHERE embedding MATCH '{query_embedding.tolist()}' ORDER BY distance LIMIT 1"
cursor = sqlite_db.execute(sql_select)
results = cursor.fetchall()

for chunk_id , text , embedding , distance in results:
    print(f"Chunk ID : {chunk_id} , Text : {text} , Distance : {distance}")

Chunk ID : 2 , Text : I am a pessimistic person. , Distance : 0.9837705492973328
